# 07: SP500 Grids — real data, recent vs history

Every result below is **loaded from disk** (`diagnosis/<grid>/<start>_<config>/metrics.json`, field `annualized_sharpe`) — nothing is hand-entered. This notebook moves from the **core** universe to the full **SP500** universe and asks whether a wider pool beats core.

**What we mean by “recent” and “historical”:**
- **Recent** = a 2024–2025 backtest. Because start-date fragility is the central risk, every recent grid is run from **5 selection start dates** and we report the **mean** across them, with the std as a stability measure.
- **Historical** = a 2015–2019 backtest from a **single start date** (see section 1).

Headline: with **2-month selection** a wider pool cannot beat core — it is too noisy. With **12-month selection** it flips: SP500 produces the strongest cell in the study (`cross_sector_slide1m_noscreen`, recent **+1.80**). But no config wins **both** windows, which is why we eventually combine two legs (section 5).

> **Limitation — universe bias.** The universes are **static lists of today's constituents** (`src/constants.py`), and prices are back-filled for the full history. So a 2015–2019 run on *today's* mega-caps (NVDA, TSLA, AVGO, META, PLTR…) uses hindsight selection + survivorship:
> - **Survivorship:** the curated core drops losers and failed peers — the historical outcome is flattered.
> - **Selection / look-ahead:** “core” is defined by *today's* large-cap status, so winner selection is known in advance.
> - **Asymmetry:** this barely touches recent (2024–25 ≈ current), but **heavily distorts historical** — which is why core looks reliable on recent yet goes strongly negative historically.
> 
> Therefore treat the **historical core-vs-SP500** comparison as directionally indicative, not a clean regime comparison.

In [1]:
import os, json
import pandas as pd

# Anchor to the repo root (parent of the notebook dir) so relative paths resolve
# regardless of the kernel's working directory.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) == 'notebooks' else None

CONFIGS = ['same_sector_slide3m_noscreen', 'same_sector_slide3m_bd7',
          'same_sector_slide1m_noscreen', 'same_sector_slide1m_bd7',
          'cross_sector_slide3m_noscreen', 'cross_sector_slide3m_bd7',
          'cross_sector_slide1m_noscreen', 'cross_sector_slide1m_bd7']

def load_grid(base, tag=''):
    """{config: [annualized_sharpe per start]} from <base>/<start>_<config>/metrics.json."""
    out = {c: [] for c in CONFIGS}
    for d in sorted(os.listdir(base)):
        for c in CONFIGS:
            if not d.endswith('_' + c):
                continue
            p = os.path.join(base, d, 'metrics.json')
            if os.path.exists(p):
                with open(p) as f:
                    out[c].append(json.load(f)['annualized_sharpe'])
    return out

def as_frame(grid):
    return pd.DataFrame({
        'config': CONFIGS,
        'sharpe': [round(pd.Series(grid[c]).mean(), 2) for c in CONFIGS],
        'std':    [round(pd.Series(grid[c]).std(ddof=0), 2) if len(grid[c]) > 1 else float('nan') for c in CONFIGS],
        'n':      [len(grid[c]) for c in CONFIGS],
    })

# ---- pct=0.045 grids (main comparison) ----
S_R_2M  = as_frame(load_grid('diagnosis/07_grid_search_sp500 (old)'))      # SP500 2m, recent (5 starts)
S_H_2M  = as_frame(load_grid('diagnosis/09_sp500_2015_2020 (old)'))        # SP500 2m, historical (1 start)
S_R_12M = as_frame(load_grid('diagnosis/7b_sp500_12m_sel_2023_2025 (old)'))   # SP500 12m, recent (5 starts)
S_H_12M = as_frame(load_grid('diagnosis/09b_sp500_12m_sel_2015_2020 (old)'))  # SP500 12m, historical (1 start)
C_R_2M  = as_frame(load_grid('diagnosis/05_grid_search_mp20 (old)'))       # core 2m, recent (5 starts)
C_H_2M  = as_frame(load_grid('diagnosis/08a_core_2015_2020 (old)'))        # core 2m, historical (1 start)
C_R_12M = as_frame(load_grid('diagnosis/6b_core_12m_sel_2023_2025 (old)'))     # core 12m, recent (5 starts)
C_H_12M = as_frame(load_grid('diagnosis/08b_core_12m_sel_2015_2020 (old)'))    # core 12m, historical (1 start)

# ---- pct=0.25 grids (used to size the combined book) ----
_S25 = 'fixed_diagnosis/_sweep_pct25/'
S_R_2M_25  = as_frame(load_grid(_S25 + '07'))      # SP500 2m recent
S_H_2M_25  = as_frame(load_grid(_S25 + '09a'))     # SP500 2m historical
S_R_12M_25 = as_frame(load_grid(_S25 + '10b'))     # SP500 12m recent
S_H_12M_25 = as_frame(load_grid(_S25 + '09b'))     # SP500 12m historical
C_R_2M_25  = as_frame(load_grid(_S25 + '06'))      # core 2m recent
C_H_2M_25  = as_frame(load_grid(_S25 + '08a'))     # core 2m historical
C_R_12M_25 = as_frame(load_grid(_S25 + '10a'))     # core 12m recent
C_H_12M_25 = as_frame(load_grid(_S25 + '08b'))     # core 12m historical



---

## 1. The 5-start methodology

Sharpe is **fragile to the start date** (it swung from −0.28 to +1.48 with a two-week shift). So a single backtest is not trustworthy. The fix: each recent grid runs the **same 8 configs from 5 selection starts**, each producing a full 2024–2025 backtest. We aggregate with the **mean Sharpe across the 5 starts** and keep the **std** as a stability measure — a config that only works from one start is untradeable.

For the historical window we used a **single start date**: over a longer trading period (5 years) the start-date instability was less pronounced, so one run is sufficient. Note the start-date field on these runs (2014) marks the **selection warm-up period**; trading begins **2015-01-01** and runs to 2019.

The code cell above loads all eight **pct=0.045** grids (SP500 and core, recent and historical, 2m and 12m). All tables in sections 3–4 use that sizing.




---

## 2. Max pairs traded per fold and per-pair sizing

### Per-pair sizing at pct=0.045 with max_pairs=20

The per-run position size and the pair count are linked. The max-pairs sweep scales `pct_per_pair` with `mp` so total exposure stays roughly constant (`mp × pct ≈ 0.9`):

| max_pairs | pct/pair | same-sector Mean Sharpe | same-sector Std | cross-sector Mean Sharpe | cross-sector Std | mean active (same / cross) |
|---:|---:|---:|---:|---:|---:|---:|
| 5 | 0.180 | 0.48 | 0.75 | 0.46 | 0.55 | 0.59 / 0.51 |
| 10 | 0.090 | 0.71 | 1.00 | -0.16 | 0.42 | 1.05 / 1.11 |
| 20 | 0.045 | 0.80 | 0.82 | 0.34 | 0.11 | 1.93 / 2.10 |
| 50 | 0.018 | 0.62 | 0.34 | 0.21 | 0.63 | 2.53 / 5.07 |

We started from a **mp=5 baseline** (section 04). On the sweep's aggregate, **mp=20** looks best — same-sector mean Sharpe peaks at **0.80**, and cross-sector is most stable at **0.34** (std 0.11). But that is only the sweep's *surface* summary. When the full 8-config grids ran, **mp=5 actually edged mp=20 on mean Sharpe**:

| Grid | Winner | Mean Sharpe | Std | Mean active trades | Mean trades |
|---|---|---:|---:|---:|---:|
| 05 (mp=5) | `same 1m ns` | **1.42** | 0.07 | 1.64 | 123.8 |
| 06 (mp=20) | `same 1m bd7` | 1.30 | 0.14 | 3.06 | 236.8 |


So **mp=20 was not kept on raw Sharpe or variance** — sections 05/06 explicitly say so. It was kept for **book thickness / deployment breadth**: a mp=5 book runs only ≈30–50 trades vs ≈130–180 at mp=20, so a single-pair idiosyncratic event (the section 03 root cause — Jaccard@5 ≈ 0.04, dropout ≈ 94%) dominates proportionally more at low `mp`. **mp=20 is the deployment-robust choice, not the raw-Sharpe max.**

To hold total exposure flat across the larger book, per-pair size dropped **0.18 → 0.045**.

### Per-pair sizing test at pct=0.25

At mp=20, on recent we rarely hold many positions at once — the best configs run **≈7 active trades** and the 3m configs ≈1–2. With **pct=0.045** (4.5% per pair), that means most of the allocated capital sits idle. The table below shows mean active trades and the resulting **gross exposure** (active trades × per-pair size, as % of equity) at both pct=0.045 and pct=0.25:

| Universe | Config | Mean active (0.045) | Gross @ 0.045 | Mean active (0.25) | Gross @ 0.25 |
|---|---|---:|---:|---:|---:|
| sp500-12m | cross 1m ns | 6.97 | 31.4% | 7.05 | 176.2% |
| sp500-12m | same 1m ns | 7.15 | 32.2% | 7.40 | 185.0% |
| sp500-12m | same 1m bd7 | 4.34 | 19.5% | 3.94 | 98.5% |
| sp500-12m | cross 1m bd7 | 3.95 | 17.8% | 3.29 | 82.2% |
| sp500-12m | cross 3m ns | 2.26 | 10.2% | 2.27 | 56.8% |
| sp500-12m | same 3m ns | 2.41 | 10.8% | 2.47 | 61.8% |
| core-12m | cross 3m ns | 1.63 | 7.3% | 1.83 | 45.8% |

**Read:** the active-trade counts barely move between pct=0.045 and pct=0.25 — the signal (which pairs, how many) is the same; only position size changes. These are **gross exposures** (active × size, as % of equity), and each pair is a long+short book. margin_behavior is **off** (None), so there is no margin/rejection guard — the backtest sizes each leg at notional = equity × pct regardless of available cash. So at **pct=0.045** the strongest configs sit at only ~31–32% gross (idle cash), but at **pct=0.25** the 1m configs go **above 100% gross** (cross 1m ns 176%, same 1m ns 185%) — that is a **levered long/short book, not cash overspent**. A live broker would require margin (or reject) above 100% gross, which this backtest does not model. The 3m configs stay under 100% (46–62%), so those alone are genuinely cured of idle cash.

In [2]:
# SP500 2m vs core, recent (5-start) and historical (single start)
m = S_R_2M[["config", "sharpe", "std"]].rename(columns={"sharpe": "sp500_recent", "std": "sp500_std"})
m["sp500_hist"] = S_H_2M["sharpe"]
m["core_recent"] = C_R_2M["sharpe"]
m["core_hist"] = C_H_2M["sharpe"]
m["d_recent"] = (m["sp500_recent"] - m["core_recent"]).round(2)
m["d_hist"] = (m["sp500_hist"] - m["core_hist"]).round(2)
m = m.sort_values("sp500_recent", ascending=False).reset_index(drop=True)
m

,config,sp500_recent,sp500_std,sp500_hist,core_recent,core_hist,d_recent,d_hist
0,same_sector_slide3m_bd7,0.74,0.82,0.24,0.47,-0.65,0.27,0.89
1,same_sector_slide1m_bd7,0.43,0.10,0.34,1.30,-0.11,-0.87,0.45
2,same_sector_slide3m_noscreen,0.40,0.77,0.24,0.50,-0.54,-0.10,0.78
3,cross_sector_slide3m_noscreen,0.27,0.36,0.62,0.54,0.24,-0.27,0.38
4,cross_sector_slide1m_noscreen,0.11,0.32,0.19,0.58,-0.04,-0.47,0.23
5,cross_sector_slide3m_bd7,0.05,0.41,0.62,0.56,0.51,-0.51,0.11
6,cross_sector_slide1m_bd7,0.05,0.05,0.19,0.63,0.50,-0.58,-0.31
7,same_sector_slide1m_noscreen,0.04,0.13,0.34,1.27,-0.08,-1.23,0.42




---

## 3. SP500 2m vs core (recent + historical)

Each window is one table: SP500 and core side by side, for recent and historical. **Recent** = mean of 5 starts; **historical** = single start.

| Config | SP500 (recent) | Core (recent) | Δ recent | SP500 (hist) | Core (hist) | Δ hist |
|---|---:|---:|---:|---:|---:|---:|
| `same 3m bd7` | +0.74 | +0.47 | +0.26 | +0.24 | -0.65 | +0.89 |
| `same 1m bd7` | +0.43 | +1.30 | -0.87 | +0.34 | -0.11 | +0.44 |
| `same 3m ns` | +0.40 | +0.50 | -0.09 | +0.24 | -0.54 | +0.78 |
| `cross 3m ns` | +0.27 | +0.54 | -0.28 | +0.62 | +0.24 | +0.37 |
| `cross 1m ns` | +0.11 | +0.58 | -0.47 | +0.19 | -0.04 | +0.23 |
| `cross 1m bd7` | +0.05 | +0.63 | -0.58 | +0.19 | +0.50 | -0.32 |
| `cross 3m bd7` | +0.05 | +0.56 | -0.51 | +0.62 | +0.51 | +0.10 |
| `same 1m ns` | +0.04 | +1.27 | -1.23 | +0.34 | -0.08 | +0.42 |

**Key findings:**
- **SP500 2m is unstable on recent and not a clear win.** Its best is `same 3m bd7` (**0.74**), but does worse than core on the 1m family (`same 1m ns` **0.04** vs core **1.27**).
- **Historically SP500 is more positive than core** on most configs (the curated core goes strongly negative here). *(Core's historical negativity is partly a survivorship artifact of today's mega-cap list — see the limitation above.)*

> Decision: no reliable 2m edge either way — the curated core remains the safer 2m choice. The > decisive gain comes only from 12m selection (section 4).

In [3]:
# SP500 12m vs core, recent (5-start) and historical (single start)
m = S_R_12M[["config", "sharpe", "std"]].rename(columns={"sharpe": "sp500_recent", "std": "sp500_std"})
m["sp500_hist"] = S_H_12M["sharpe"]
m["core_recent"] = C_R_12M["sharpe"]
m["core_hist"] = C_H_12M["sharpe"]
m["d_recent"] = (m["sp500_recent"] - m["core_recent"]).round(2)
m["d_hist"] = (m["sp500_hist"] - m["core_hist"]).round(2)
m = m.sort_values("sp500_recent", ascending=False).reset_index(drop=True)
m

,config,sp500_recent,sp500_std,sp500_hist,core_recent,core_hist,d_recent,d_hist
0,cross_sector_slide1m_noscreen,1.80,0.41,0.31,0.82,-0.43,0.98,0.74
1,same_sector_slide1m_noscreen,1.63,0.04,0.43,0.34,-0.80,1.29,1.23
2,same_sector_slide1m_bd7,1.48,0.07,0.38,0.37,0.13,1.11,0.25
3,cross_sector_slide1m_bd7,1.38,0.48,0.31,0.52,0.34,0.86,-0.03
4,cross_sector_slide3m_noscreen,1.06,0.30,-0.27,0.42,-0.63,0.64,0.36
5,same_sector_slide3m_bd7,1.03,0.48,0.35,0.14,-0.34,0.89,0.69
6,same_sector_slide3m_noscreen,1.01,0.86,0.35,0.28,-0.91,0.73,1.26
7,cross_sector_slide3m_bd7,0.78,0.55,-0.27,0.31,-0.00,0.47,-0.27




---

## 4. SP500 12m vs core (recent + historical)

With 12m selection the picture **inverts** — a wide universe becomes a benefit:

| Config | SP500 (recent) | Core (recent) | Δ recent | SP500 (hist) | Core (hist) | Δ hist |
|---|---:|---:|---:|---:|---:|---:|
| `cross 1m ns` | +1.80 | +0.82 | +0.98 | +0.31 | -0.43 | +0.73 |
| `same 1m ns` | +1.63 | +0.34 | +1.28 | +0.43 | -0.80 | +1.22 |
| `same 1m bd7` | +1.48 | +0.37 | +1.11 | +0.38 | +0.13 | +0.25 |
| `cross 1m bd7` | +1.38 | +0.52 | +0.86 | +0.31 | +0.34 | -0.03 |
| `cross 3m ns` | +1.06 | +0.42 | +0.64 | -0.27 | -0.63 | +0.36 |
| `same 3m bd7` | +1.03 | +0.14 | +0.88 | +0.35 | -0.34 | +0.69 |
| `same 3m ns` | +1.01 | +0.28 | +0.73 | +0.35 | -0.91 | +1.26 |
| `cross 3m bd7` | +0.78 | +0.31 | +0.46 | -0.27 | -0.00 | -0.27 |

**Key findings:**
- **Recent:** SP500 beats core on **every** config; `cross 1m noscreen` SP500 **1.80** — the strongest cell in the whole study.
- **Historical:** the gain is broad but not universal — SP500 beats core on **6 of 8** configs; `cross 1m noscreen` is **+0.31** vs core **-0.43**. `cross 3m` goes negative historically (`cross 3m ns`, **-0.27**). *(Core's historical negativity is partly a survivorship artifact of today's mega-cap list — see the limitation above.)*

> Decision: the wider universe is a *benefit* when selection is good — the opposite of the 2m > result.



---

## 5. No config wins both windows — hence the combined book

Look at the SP500 12m configs across both windows:

| Config | Recent (12m) | Historical (12m) |
|---|---:|---:|
| `cross 1m ns` | +1.80 | +0.31 |
| `same 1m ns` | +1.63 | +0.43 |
| `cross 3m ns` | +1.06 | -0.27 |

**Why a combined book:**
- `cross 1m ns` is **#1 recent** (**+1.80**) but is not the historical leader (**+0.31**).
- `same 1m ns` is the **#1 historical** (**+0.43**) but trails recent (**+1.63** vs 1.80).
- `cross 3m ns` is solid recent (**+1.06**) yet **negative historically** (−0.27).

A single config cannot be the best on **both** windows — the “winner” flips between the 1m family (strong recent) and cross-3m (weak historical). The resolution is a **two-leg combined book** that weights a strong-recent leg against a historical-robust leg rather than betting on one config. See [notebook 08](08_combined_momentum_book.ipynb) for how the two legs are combined and sized.

In [4]:
# SP500 vs core, 2m selection, at pct=0.25
m = S_R_2M_25[["config", "sharpe"]].rename(columns={"sharpe": "sp500_recent"})
m["core_recent"] = C_R_2M_25["sharpe"]
m["sp500_hist"] = S_H_2M_25["sharpe"]
m["core_hist"] = C_H_2M_25["sharpe"]
m["d_recent"] = (m["sp500_recent"] - m["core_recent"]).round(2)
m["d_hist"] = (m["sp500_hist"] - m["core_hist"]).round(2)
m = m.sort_values("sp500_recent", ascending=False).reset_index(drop=True)
m

,config,sp500_recent,core_recent,sp500_hist,core_hist,d_recent,d_hist
0,same_sector_slide3m_noscreen,0.42,0.39,0.33,-0.41,0.03,0.74
1,cross_sector_slide3m_noscreen,0.38,0.62,0.57,0.24,-0.24,0.33
2,cross_sector_slide1m_noscreen,0.37,0.63,0.14,-0.04,-0.26,0.18
3,same_sector_slide3m_bd7,0.29,0.44,0.37,-0.75,-0.15,1.12
4,same_sector_slide1m_noscreen,0.12,1.02,0.48,-0.09,-0.90,0.57
5,same_sector_slide1m_bd7,-0.39,1.05,-0.11,-0.10,-1.44,-0.01
6,cross_sector_slide3m_bd7,-0.40,0.57,0.75,0.47,-0.97,0.28
7,cross_sector_slide1m_bd7,-0.48,0.65,0.43,0.49,-1.13,-0.06


In [5]:
# SP500 vs core, 12m selection, at pct=0.25
m = S_R_12M_25[["config", "sharpe"]].rename(columns={"sharpe": "sp500_recent"})
m["core_recent"] = C_R_12M_25["sharpe"]
m["sp500_hist"] = S_H_12M_25["sharpe"]
m["core_hist"] = C_H_12M_25["sharpe"]
m["d_recent"] = (m["sp500_recent"] - m["core_recent"]).round(2)
m["d_hist"] = (m["sp500_hist"] - m["core_hist"]).round(2)
m = m.sort_values("sp500_recent", ascending=False).reset_index(drop=True)
m

,config,sp500_recent,core_recent,sp500_hist,core_hist,d_recent,d_hist
0,cross_sector_slide1m_noscreen,1.76,0.83,0.50,-0.52,0.93,1.02
1,same_sector_slide1m_noscreen,1.59,0.33,0.40,-0.79,1.26,1.19
2,cross_sector_slide1m_bd7,1.07,0.39,0.56,0.34,0.68,0.22
3,same_sector_slide3m_noscreen,1.02,0.26,0.35,-0.90,0.76,1.25
4,cross_sector_slide3m_noscreen,1.02,0.41,-0.11,-0.63,0.61,0.52
5,same_sector_slide1m_bd7,0.83,0.30,0.45,0.13,0.53,0.32
6,cross_sector_slide3m_bd7,0.69,0.15,0.31,0.01,0.54,0.30
7,same_sector_slide3m_bd7,0.61,0.09,0.17,-0.34,0.52,0.51




---

## 6. Range of run at pct=0.25

The sections 3–4 numbers are at **pct=0.045** (4.5% of equity per pair). The combined book instead sizes at **pct=0.25**, so here is the same comparison re-run at that sizing (loaded from ixed_diagnosis/_sweep_pct25/). Note the pct=0.25 runs use section dirs 06/07 (2m) and 10a/10b (12m), where **10a = core** and **10b = sp500**.

**5a. SP500 2m — recent vs historical (pct=0.25):**

| Config | Recent (mean 5) | Historical (single start) |
|---|---:|---:|
| same 3m ns | +0.42 | +0.33 |
| cross 3m ns | +0.38 | +0.57 |
| cross 1m ns | +0.37 | +0.14 |
| same 3m bd7 | +0.29 | +0.37 |
| same 1m ns | +0.12 | +0.48 |
| same 1m bd7 | -0.39 | -0.11 |
| cross 3m bd7 | -0.40 | +0.75 |
| cross 1m bd7 | -0.48 | +0.43 |

**5b. SP500 12m — recent vs historical (pct=0.25):**

| Config | Recent (mean 5) | Historical (single start) |
|---|---:|---:|
| cross 1m ns | +1.76 | +0.50 |
| same 1m ns | +1.59 | +0.40 |
| cross 1m bd7 | +1.07 | +0.56 |
| same 3m ns | +1.02 | +0.35 |
| cross 3m ns | +1.02 | -0.11 |
| same 1m bd7 | +0.83 | +0.45 |
| cross 3m bd7 | +0.69 | +0.31 |
| same 3m bd7 | +0.61 | +0.17 |

**5c. SP500 vs core, 2m — recent + historical (pct=0.25):**

| Config | SP500 (recent) | Core (recent) | Δ recent | SP500 (hist) | Core (hist) | Δ hist |
|---|---:|---:|---:|---:|---:|---:|
| same 3m ns | +0.42 | +0.39 | +0.02 | +0.33 | -0.41 | +0.74 |
| cross 3m ns | +0.38 | +0.62 | -0.24 | +0.57 | +0.24 | +0.32 |
| cross 1m ns | +0.37 | +0.63 | -0.26 | +0.14 | -0.04 | +0.18 |
| same 3m bd7 | +0.29 | +0.44 | -0.15 | +0.37 | -0.75 | +1.13 |
| same 1m ns | +0.12 | +1.02 | -0.90 | +0.48 | -0.09 | +0.58 |
| same 1m bd7 | -0.39 | +1.05 | -1.44 | -0.11 | -0.10 | -0.01 |
| cross 3m bd7 | -0.40 | +0.57 | -0.97 | +0.75 | +0.47 | +0.27 |
| cross 1m bd7 | -0.48 | +0.65 | -1.13 | +0.43 | +0.49 | -0.06 |

**5d. SP500 vs core, 12m — recent + historical (pct=0.25):**

| Config | SP500 (recent) | Core (recent) | Δ recent | SP500 (hist) | Core (hist) | Δ hist |
|---|---:|---:|---:|---:|---:|---:|
| cross 1m ns | +1.76 | +0.83 | +0.93 | +0.50 | -0.52 | +1.02 |
| same 1m ns | +1.59 | +0.33 | +1.26 | +0.40 | -0.79 | +1.19 |
| cross 1m bd7 | +1.07 | +0.39 | +0.68 | +0.56 | +0.34 | +0.21 |
| same 3m ns | +1.02 | +0.26 | +0.76 | +0.35 | -0.90 | +1.25 |
| cross 3m ns | +1.02 | +0.41 | +0.60 | -0.11 | -0.63 | +0.51 |
| same 1m bd7 | +0.83 | +0.30 | +0.53 | +0.45 | +0.13 | +0.32 |
| cross 3m bd7 | +0.69 | +0.15 | +0.54 | +0.31 | +0.01 | +0.30 |
| same 3m bd7 | +0.61 | +0.09 | +0.52 | +0.17 | -0.34 | +0.50 |

**Key findings:**
- At **pct=0.25** the SP500-12m recent winner (cross 1m ns) is **+1.76** — still comfortably above core (**+0.83**), so the wider universe remains the stronger recent leg even at higher sizing.
- **SP500 beats core on recent** across every config; historically too on most (SP500 cross 1m ns hist **+0.50** vs core **-0.52**). The core 12m recent config that trails here does **not** hold historically (core 12m hist same 3m ns **-0.90**).
- **The 1m family is robust to sizing on both windows**, reinforcing the choice to combine 1m legs across windows.



---

## 7. Where this leaves us

- **Core is better for 2m selection on recent, worse on historical; SP500 is generally better for 12m selection on both periods.** The selection window, not the universe, is the binding choice.
- No single config wins both windows (section 5), so we move to a **two-leg combined book** rather than a single best config.
- `sp500-12m / cross_sector_slide1m_noscreen` is the strongest recent cell in the study (**+1.80** at pct=0.045), but the leader changes at pct=0.25 (section 6).
- Every number above is traceable to `diagnosis/**/metrics.json` and `fixed_diagnosis/_sweep_pct25/**` via the loader code cells.
- The combined two-leg book is built and sized in [notebook 08](08_combined_momentum_book.ipynb).